Cleaning churn analysis 

In [1]:
import pandas as pd

churn_analysis = pd.read_csv(r"C:\Users\chase\OneDrive\Desktop\Adventure Works Project\Raw csvs\churn_analysis.csv")
churn_analysis.info()
churn_analysis.head()

<class 'pandas.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   order_quarter          13 non-null     str    
 1   new_customers          13 non-null     int64  
 2   retained_customers     13 non-null     int64  
 3   reactivated_customers  13 non-null     int64  
 4   churned_customers      12 non-null     float64
dtypes: float64(1), int64(3), str(1)
memory usage: 652.0 bytes


,order_quarter,new_customers,retained_customers,reactivated_customers,churned_customers
0,2022-04-01,264,0,0,155.0
1,2022-07-01,545,109,0,484.0
2,2022-10-01,603,170,1,603.0
3,2023-01-01,589,171,14,634.0
4,2023-04-01,798,140,14,722.0


Must change "order_quarter" from str to datetime. "churned_customers" stays as float instead of int because of the null value in the most recent quarter.

In [2]:
churn_analysis["order_quarter"] = pd.to_datetime(churn_analysis["order_quarter"])

churn_analysis.info()

<class 'pandas.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   order_quarter          13 non-null     datetime64[us]
 1   new_customers          13 non-null     int64         
 2   retained_customers     13 non-null     int64         
 3   reactivated_customers  13 non-null     int64         
 4   churned_customers      12 non-null     float64       
dtypes: datetime64[us](1), float64(1), int64(3)
memory usage: 652.0 bytes


Changed to long format so that customer_status can be a dimension instead of having each status be separate measure. This will make creating visualizations in tableau easier. 

In [5]:
churn_analysis_long = pd.melt(
    churn_analysis,
    id_vars = "order_quarter",
    value_vars = ["new_customers", "retained_customers", "reactivated_customers", "churned_customers"],
    var_name = "customer_status",
    value_name = "customer_count"
)

churn_analysis_long = churn_analysis_long.sort_values("order_quarter")

churn_analysis_long.head()

,order_quarter,customer_status,customer_count
0,2022-04-01,new_customers,264.0
26,2022-04-01,reactivated_customers,0.0
39,2022-04-01,churned_customers,155.0
13,2022-04-01,retained_customers,0.0
1,2022-07-01,new_customers,545.0


Created "is_measurable" to make it clear that churned customers couldn't be determined in the most recent quarter and to be able to filter that null out when making visualizations. 

In [ ]:
churn_analysis_long["is_measurable"] = ~(
    (churn_analysis_long["customer_status"] == "churned_customers")
    & (churn_analysis_long["customer_count"].isna()))

churn_analysis_long["is_measurable"].value_counts()


is_measurable
True     51
False     1
Name: count, dtype: int64

In [14]:
churn_analysis_long.to_csv(r"C:\Users\chase\OneDrive\Desktop\Adventure Works Project\Clean csvs\churn_analysis_cleaned.csv", index = False)